In [1]:
import pandas as pd
import numpy as np

# load dataset 

normal_dataset_path = '../dataset/clean/normal/normal.csv'

attack_0rtt_dataset_path = '../dataset/clean/tls/attack_0rtt_dataset.csv'
attack_heartbleed_dataset_path = '../dataset/clean/tls/attack_heartbleed_dataset.csv'


attack_cert_probe_dataset_path = '../dataset/clean/probe/attack_cert_probe_dataset.csv'
attack_crypto_probe_dataset_path = '../dataset/clean/probe/attack_crypto_probe_dataset.csv'
attack_cve_probe_dataset_path = '../dataset/clean/probe/attack_cve_probe_dataset.csv'
attack_protocol_probe_dataset_path = '../dataset/clean/probe/attack_protocol_probe_dataset.csv'

attack_goldeneye_dataset_path = '../dataset/clean/dos/attack_goldeneye_dataset.csv'
attack_hulk_dataset_path = '../dataset/clean/dos/attack_hulk_dataset.csv'
attack_rst_flood_dataset_path = '../dataset/clean/dos/attack_rst_flood_dataset.csv'
attack_slowloris_dataset_path = '../dataset/clean/dos/attack_slowloris_dataset.csv'
attack_sync_flood_dataset_path = '../dataset/clean/dos/attack_sync_flood_dataset.csv'
attack_tcp_ack_dataset_path = '../dataset/clean/dos/attack_tcp_ack_dataset.csv'
attack_tors_dataset_path = '../dataset/clean/dos/attack_tors_dataset.csv'
attack_udp_dataset_path = '../dataset/clean/dos/attack_udp_dataset.csv'




normal_dataset = pd.read_csv(normal_dataset_path)

attack_0rtt_dataset = pd.read_csv(attack_0rtt_dataset_path)
attack_cert_probe_dataset = pd.read_csv(attack_cert_probe_dataset_path)
attack_crypto_probe_dataset = pd.read_csv(attack_crypto_probe_dataset_path)
attack_cve_probe_dataset = pd.read_csv(attack_cve_probe_dataset_path)
attack_goldeneye_dataset = pd.read_csv(attack_goldeneye_dataset_path)
attack_heartbleed_dataset = pd.read_csv(attack_heartbleed_dataset_path)
attack_hulk_dataset = pd.read_csv(attack_hulk_dataset_path)
attack_protocol_probe_dataset = pd.read_csv(attack_protocol_probe_dataset_path)
attack_rst_flood_dataset = pd.read_csv(attack_rst_flood_dataset_path)
attack_slowloris_dataset = pd.read_csv(attack_slowloris_dataset_path)
attack_sync_flood_dataset = pd.read_csv(attack_sync_flood_dataset_path)
attack_tcp_ack_dataset = pd.read_csv(attack_tcp_ack_dataset_path)
attack_tors_dataset = pd.read_csv(attack_tors_dataset_path)
attack_udp_dataset = pd.read_csv(attack_udp_dataset_path)


# Randomly sample data

# Normal
normal_dataset = normal_dataset.sample(n=49000, random_state=42) 

# TLS (use full)
tls_dataset = pd.concat(
    [
        attack_0rtt_dataset,
        attack_heartbleed_dataset
    ],
    axis=0,
    ignore_index=True
)

# Probe 
probe_dataset = pd.concat(
    [
        attack_cert_probe_dataset.sample(n=1000, random_state=42),
        attack_crypto_probe_dataset.sample(n=1000, random_state=42) ,
        attack_cve_probe_dataset.sample(n=2000, random_state=42) ,
        attack_protocol_probe_dataset.sample(n=1000, random_state=42) 
    ],
    axis=0,
    ignore_index=True
)

# Dos
dos_dataset = pd.concat(
    [
        attack_goldeneye_dataset.sample(n=1000, random_state=42),
        attack_hulk_dataset.sample(n=1000, random_state=42),
        attack_rst_flood_dataset,
        attack_slowloris_dataset,
        attack_sync_flood_dataset.sample(n=1000, random_state=42),
        attack_tcp_ack_dataset.sample(n=1000, random_state=42),
        attack_tors_dataset.sample(n=1000, random_state=42),
        attack_udp_dataset
    ],
    axis=0,
    ignore_index=True
)



In [2]:
normal_df = normal_dataset
attack_df = pd.concat([tls_dataset, probe_dataset, dos_dataset], ignore_index=True)

# Combine all
dataset_df = pd.concat([normal_df, attack_df], ignore_index=True)

#  Shuffle the data
dataset_df = dataset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Result
print("Combined shape:", dataset_df.shape)
print(dataset_df['label'].value_counts())

Combined shape: (59380, 77)
label
normal    49000
ddos       5264
probe      5000
tls         116
Name: count, dtype: int64


# IDS Autoencoder

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score, f1_score
)

# ========= 1) Data =========
X = dataset_df.drop(columns=['label']).values.astype(np.float32)   # (N, 77)
y_str = dataset_df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_str).astype(np.int64)
classes = le.classes_
num_classes = len(classes)
print("Classes:", classes)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Stratified 70/10/20 split
X_tr_full, X_te, y_tr_full, y_te = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_full, y_tr_full, test_size=0.125, random_state=42, stratify=y_tr_full
)  # -> 70/10/20

# Tensors / loaders
X_tr_t  = torch.tensor(X_tr, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_te_t  = torch.tensor(X_te,  dtype=torch.float32)
y_tr_t  = torch.tensor(y_tr,  dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_te_t  = torch.tensor(y_te,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),   batch_size=256, shuffle=False)

# ========= 2) Class weights (from TRAIN only) =========
unique, counts = np.unique(y_tr, return_counts=True)
inv = {k: 1.0/v for k, v in dict(zip(unique, counts)).items()}
scale = np.mean(list(inv.values()))
class_weights = torch.tensor([inv[k]/scale for k in sorted(inv.keys())], dtype=torch.float32)

# ========= 3) Supervised Autoencoder =========
class SupAutoEncoder(nn.Module):
    def __init__(self, input_dim=77, enc_dims=(128, 64), latent_dim=16,
                 dec_dims=(64, 128), num_classes=4, dropout=0.2):
        super().__init__()
        # Encoder
        layers = []
        prev = input_dim
        for h in enc_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, latent_dim)]  # latent
        self.encoder = nn.Sequential(*layers)

        # Classifier head (on latent)
        self.classifier = nn.Sequential(
            nn.ReLU(),                      # a little nonlinearity post-latent
            nn.Dropout(dropout),
            nn.Linear(latent_dim, num_classes)
        )

        # Decoder (mirror-ish)
        layers = []
        prev = latent_dim
        for h in dec_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*layers)

    def forward(self, x):
        z = self.encoder(x)            # (B, latent_dim)
        logits = self.classifier(z)    # (B, C)
        x_hat = self.decoder(z)        # (B, input_dim)
        return logits, x_hat, z

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SupAutoEncoder(
    input_dim=X_tr.shape[1], enc_dims=(128, 64), latent_dim=16,
    dec_dims=(64, 128), num_classes=num_classes, dropout=0.2
).to(device)

# ========= 4) Loss / Optimizer =========
ce_criterion   = nn.CrossEntropyLoss(weight=class_weights.to(device))
recon_criterion = nn.MSELoss()  # reconstruction loss

# Balance between classification and reconstruction:
lambda_recon = 0.2   # weight of reconstruction (tune 0.05–0.5)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 80
best_val_bal_acc = 0.0

# ========= 5) Train / Validate =========
for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, x_hat, _ = model(xb)
        loss_cls = ce_criterion(logits, yb)
        loss_rec = recon_criterion(x_hat, xb)
        loss = loss_cls + lambda_recon * loss_rec
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    # Validation
    model.eval()
    val_true, val_pred = [], []
    val_cls_loss, val_rec_loss = 0.0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, x_hat, _ = model(xb)
            val_cls_loss += ce_criterion(logits, yb).item()
            val_rec_loss += recon_criterion(x_hat, xb).item()
            preds = logits.argmax(dim=1)
            val_true.extend(yb.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = accuracy_score(val_true, val_pred)
    val_bal_acc = balanced_accuracy_score(val_true, val_pred)

    if val_bal_acc > best_val_bal_acc:
        best_val_bal_acc = val_bal_acc
        torch.save(model.state_dict(), "best_supae.pth")

    print(f"Epoch {epoch:03d} | TrainLoss {tr_loss/len(train_loader):.4f} "
          f"| ValCls {val_cls_loss/len(val_loader):.4f} "
          f"| ValRec {val_rec_loss/len(val_loader):.4f} "
          f"| ValAcc {val_acc:.4f} | ValBalanced {val_bal_acc:.4f}")

# ========= 6) Test (classifier output) =========
model.load_state_dict(torch.load("best_supae.pth", map_location=device))
model.eval()
y_true, y_pred, recon_errs = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits, x_hat, _ = model(xb)
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
        y_true.extend(yb.numpy())
        # per-sample reconstruction error (MSE)
        recon_errs.extend(((x_hat - xb)**2).mean(dim=1).cpu().numpy())

print("TEST accuracy:", accuracy_score(y_true, y_pred))
print("TEST balanced_accuracy:", balanced_accuracy_score(y_true, y_pred))
print("TEST macro F1:", f1_score(y_true, y_pred, average="macro"))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=classes))

# (Optional) you can inspect reconstruction error distribution per predicted/true class:
# import numpy as np
# recon_errs = np.array(recon_errs)

Classes: ['ddos' 'normal' 'probe' 'tls']
Epoch 001 | TrainLoss 0.6497 | ValCls 0.3583 | ValRec 0.3130 | ValAcc 0.9872 | ValBalanced 0.8211
Epoch 002 | TrainLoss 0.3529 | ValCls 0.2865 | ValRec 0.2515 | ValAcc 0.9795 | ValBalanced 0.8421
Epoch 003 | TrainLoss 0.3072 | ValCls 0.2869 | ValRec 0.2287 | ValAcc 0.9827 | ValBalanced 0.8427
Epoch 004 | TrainLoss 0.2652 | ValCls 0.2309 | ValRec 0.1999 | ValAcc 0.9899 | ValBalanced 0.8449
Epoch 005 | TrainLoss 0.2566 | ValCls 0.2364 | ValRec 0.1800 | ValAcc 0.9926 | ValBalanced 0.8457
Epoch 006 | TrainLoss 0.2277 | ValCls 0.2403 | ValRec 0.1611 | ValAcc 0.9933 | ValBalanced 0.8459
Epoch 007 | TrainLoss 0.2457 | ValCls 0.2272 | ValRec 0.1577 | ValAcc 0.9847 | ValBalanced 0.9472
Epoch 008 | TrainLoss 0.2064 | ValCls 0.1801 | ValRec 0.1565 | ValAcc 0.9874 | ValBalanced 0.9688
Epoch 009 | TrainLoss 0.2107 | ValCls 0.1919 | ValRec 0.1436 | ValAcc 0.9757 | ValBalanced 0.9445
Epoch 010 | TrainLoss 0.1902 | ValCls 0.1927 | ValRec 0.1321 | ValAcc 0.9757 

/tmp/ipykernel_1352553/3536805100.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_supae.pth", map_location=device))
